# physics_replay on Colab: Panda → UR5e pick-and-place retargeting

This notebook clones the public [`trajectory-retargeting-starter`](https://github.com/loversky02/trajectory-retargeting-starter) repository and re-runs the reference demo:

- the 20 tests;
- the demo (Panda source run plus four UR5e target runs, with videos);
- replay of the committed 10 Hz example episode;
- the two seeded benchmarks;
- a check of each benchmark against the committed results.

A **CPU runtime is enough**. Without a GPU, MuJoCo renders through OSMesa, which is slower but only matters for the videos. Rough timings on a laptop CPU: demo about 1 min, dev benchmark about 2 min, held-out benchmark about 3–4 min. Colab CPUs are usually slower.

Run the cells from top to bottom. Every command runs in a subprocess, so the pinned `numpy` can be installed without restarting the runtime.


## 1. Get the public code

The next cell clones the public repository's `main` branch directly. No ZIP upload, GitHub account, or authentication is required.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

GIT_URL = "https://github.com/loversky02/trajectory-retargeting-starter.git"
GIT_BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
# Locally the notebook sits in physics_replay/, so the repository root is its parent.
REPO = Path("/content/trajectory-retargeting-starter") if IN_COLAB else Path.cwd().resolve().parent


def sh(command):
    """Run a shell command, stream its output into the notebook, and stop on failure."""
    process = subprocess.Popen(command, shell=True, cwd=REPO if REPO.exists() else None,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"command failed with exit code {process.returncode}: {command}")


if not (REPO / "physics_replay" / "__main__.py").exists():
    if not IN_COLAB:
        raise FileNotFoundError("run this notebook from the physics_replay/ folder of the repository")
    REPO.parent.mkdir(parents=True, exist_ok=True)
    sh(f"git clone --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO}")
os.chdir(REPO)
assert (REPO / "config" / "physics-replay" / "pick-place-v1.json").exists(), "config/physics-replay is missing from the public repository"
print("repository root:", REPO)


## 2. Install the pinned packages and pick a renderer

`requirements.txt` pins the versions the committed results were produced with. If a pin has no wheel for this Python, the cell falls back to the latest versions and says so. The manifest of every run records the versions actually used.

In [ ]:
PY = sys.executable
if IN_COLAB:  # locally, set up the environment as in README.md instead
    try:
        sh(f"{PY} -m pip install -q -r physics_replay/requirements.txt")
    except RuntimeError:
        print("pinned install failed; installing unpinned versions (results may differ slightly)")
        sh(f"{PY} -m pip install -q mujoco numpy matplotlib imageio imageio-ffmpeg pillow pytest")
    if shutil.which("nvidia-smi"):
        icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
        if not icd.exists():
            icd.write_text('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
        os.environ["MUJOCO_GL"] = "egl"
    else:
        sh("apt-get -qq update > /dev/null && apt-get -qq install -y libosmesa6-dev > /dev/null")
        sh(f"{PY} -m pip install -q pyopengl")
        os.environ["MUJOCO_GL"] = "osmesa"
        os.environ["PYOPENGL_PLATFORM"] = "osmesa"

RENDER_CHECK = ("import os, mujoco; model = mujoco.MjModel.from_xml_string('<mujoco/>'); "
                "mujoco.Renderer(model, 64, 64).render(); "
                "print('rendering works, MUJOCO_GL =', os.environ.get('MUJOCO_GL', 'default'))")
try:
    sh(f'{PY} -c "{RENDER_CHECK}"')
    VIDEO = ""
except RuntimeError:
    print("rendering is unavailable here; the demo will run without videos")
    VIDEO = "--no-video"

## 3. Fetch the robot models

Downloads the Panda, UR5e and Robotiq 2F-85 from one pinned MuJoCo Menagerie commit (117 files, about 66 MB) and writes a SHA-256 receipt. Every later run checks that receipt offline. The cell makes one call to the GitHub API; if it fails with HTTP 403, the shared Colab IP has hit GitHub's rate limit, so wait a few minutes and run it again.

In [ ]:
sh(f"{PY} -m physics_replay fetch-models")

## 4. Run the tests

15 contract tests (transforms, schema, segmentation, statistics, config) and 5 physics integration tests.

In [ ]:
sh(f"{PY} -m pytest -q physics_replay/tests")

## 5. The demo

The Panda source demo, then the same episode retargeted onto the UR5e in a scene it never saw, by the three methods and the scripted upper bound. Each line of the printed JSON is one target run.

In [ ]:
RUNS = REPO / "data" / "physics-replay" / "runs" / "colab"
sh(f"{PY} -m physics_replay demo --out {RUNS / 'demo'} {VIDEO}")

In [ ]:
import base64
from IPython.display import HTML, Image, Markdown, display


def show_video(path, width=960):
    data = base64.b64encode(Path(path).read_bytes()).decode()
    display(HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))


demo = RUNS / "demo"
if (demo / "comparison.mp4").exists():
    show_video(demo / "comparison.mp4")
display(Image(str(demo / "target_object_relative_trimmed_tracking.png"), width=900))

## 6. Replay the committed example episode

`examples/panda_pick_place_10hz.json` is the source demo downsampled to 10 Hz, with the object pose missing (`null`) in 36 of 102 frames. Retargeting needs the object pose only before the lift, so the replay should still succeed.

In [ ]:
sh(f"{PY} -m physics_replay replay --episode physics_replay/examples/panda_pick_place_10hz.json "
   f"--method object_relative_trimmed --out {RUNS / 'replay-example'} {VIDEO}")

## 7. The benchmarks

Both use the settings behind the committed results: the dev run (config seed, 50 layouts) and the held-out run (seed 7, 100 layouts). Every method runs on the same sampled layouts. To try other layouts, change `--seed` or `--configs`.

In [ ]:
sh(f"{PY} -m physics_replay benchmark --out {RUNS / 'bench-dev-n50'}")

In [ ]:
sh(f"{PY} -m physics_replay benchmark --seed 7 --configs 100 --out {RUNS / 'bench-heldout-seed7-n100'}")

In [ ]:
for name in ("bench-dev-n50", "bench-heldout-seed7-n100"):
    display(Markdown((RUNS / name / "summary.md").read_text()))
    display(Image(str(RUNS / name / "success_rates.png"), width=700))

## 8. Compare with the committed results

Runs are deterministic on one machine. Across machines, floating-point differences in MuJoCo or NumPy can flip a trial that sits near a threshold. This cell therefore reports how many trials agree with `physics_replay/results/` and lists any that differ; it does not assert equality.

In [ ]:
import numpy as np
import json


def flat(layout):
    """Every number in a layout, in a fixed order."""
    return [v for part in layout.values() for v in part.values()]


def trials(path):
    rows = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    return {(r["layout_index"], r["method"]): r for r in rows}


for name in ("bench-dev-n50", "bench-heldout-seed7-n100"):
    committed = trials(REPO / "physics_replay" / "results" / name / "trials.jsonl")
    fresh = trials(RUNS / name / "trials.jsonl")
    # Layouts are sampled with floating-point math, so another CPU can differ in the last bit.
    same_layouts = all(np.allclose(flat(committed[key]["layout"]), flat(fresh[key]["layout"]), rtol=0, atol=1e-12)
                       for key in committed if key in fresh)
    differ = [key for key in committed if key not in fresh
              or (committed[key]["metrics"]["success"], committed[key]["metrics"]["failure_reason"])
              != (fresh[key]["metrics"]["success"], fresh[key]["metrics"]["failure_reason"])]
    print(f"{name}: {len(committed) - len(differ)}/{len(committed)} trials agree with the committed results; "
          f"layouts match to 1e-12: {same_layouts}")
    for layout_index, method in differ:
        before = committed[(layout_index, method)]["metrics"]
        after = fresh.get((layout_index, method), {}).get("metrics", {})
        print(f"  layout {layout_index} {method}: committed {before['failure_reason'] or 'success'}, "
              f"this run {after.get('failure_reason') or ('success' if after.get('success') else 'missing')}")

## 9. Download the outputs

Zips everything this notebook produced: the episodes, plans, tracking plots, videos, manifests and benchmark logs.

In [ ]:
archive = shutil.make_archive(str(RUNS.parent / "colab-runs"), "zip", RUNS)
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(archive)